# MSI Combinational Blocks — Routing Bits With Select Lines

Medium-scale blocks route data under the control of select lines: a **multiplexer** picks one of many inputs, a **decoder** activates one of many outputs, and **encoders** do the reverse. This notebook draws each block as a schematic with the **active path lit by the current select value**, so you can see which line is steering the data.

$$Y = \sum_{i} \big(\text{sel}=i\big)\cdot D_i$$


In [1]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, Polygon
import ipywidgets as widgets
from IPython.display import display
%matplotlib inline

plt.rcParams.update({'figure.dpi':110,'axes.spines.top':False,'axes.spines.right':False,'font.size':9})
ON, OFF = '#c0392b', '#b0b0b0'
def wcol(b): return ON if b else OFF
def wlw(b):  return 2.6 if b else 1.1
def selcol(active): return '#8e44ad' if active else '#cccccc'
print('primitives ready')


primitives ready


## Multiplexer — One of N Inputs Reaches the Output

An $N{:}1$ MUX has $\log_2 N$ select lines. The trapezoid below narrows from inputs to a single output; the **selected input wire and its path through the block light up**, and every other input is greyed. Change the select value to steer a different line through.

$$Y = D_{\text{sel}}, \qquad \text{sel} \in \{0,\dots,N-1\}$$


In [2]:
def draw_mux(D_str, sel):
    N = 4
    D = [int(c) for c in D_str.ljust(N,'0')[:N]]
    Y = D[sel]
    fig, ax = plt.subplots(figsize=(7,4)); ax.set_xlim(0,8); ax.set_ylim(0,5); ax.axis('off')
    # trapezoid body
    body = Polygon([(3,0.5),(5,1.5),(5,3.5),(3,4.5)], closed=True, fc='#eef2f7', ec='#34495e', lw=1.6, zorder=2)
    ax.add_patch(body)
    ax.text(4,2.5,f'{N}:1\nMUX',ha='center',va='center',fontsize=9,weight='bold',zorder=3)
    ys = np.linspace(4.0,1.0,N)
    for i in range(N):
        active = (i==sel)
        ax.scatter([0.6],[ys[i]],s=40,color=wcol(D[i]),zorder=4)
        ax.text(0.3,ys[i],f'D{i}={D[i]}',ha='right',va='center',color=wcol(D[i]),fontsize=9,weight='bold')
        # wire from input to body edge
        xe = 3 + (4.5-ys[i])*0 if ys[i]>2.5 else 3   # left edge x is 3
        col = wcol(D[i]) if active else '#dddddd'
        lw = wlw(D[i]) if active else 1.0
        ax.plot([0.6,3.0],[ys[i],ys[i]],color=col,lw=lw,zorder=1)
        if active:
            ax.plot([3.0,4.0],[ys[i],2.5],color=wcol(D[i]),lw=wlw(D[i]),zorder=3)
    # output
    ax.plot([5.0,6.6],[2.5,2.5],color=wcol(Y),lw=wlw(Y),zorder=3)
    ax.scatter([6.6],[2.5],s=44,color=wcol(Y),zorder=4)
    ax.text(6.85,2.5,f'Y={Y}',ha='left',va='center',color=wcol(Y),fontsize=10,weight='bold')
    # select lines
    nbits=int(np.log2(N))
    sbits=[(sel>>b)&1 for b in range(nbits)]
    ax.text(4,0.2,'sel = '+format(sel,f'0{nbits}b')+f' (={sel})',ha='center',fontsize=9,color='#8e44ad',weight='bold')
    ax.annotate('',xy=(4,0.9),xytext=(4,0.4),arrowprops=dict(arrowstyle='->',color='#8e44ad',lw=1.6))
    ax.set_title(f'4:1 MUX  --  D{sel} routed to output',fontsize=10)
    plt.tight_layout(); plt.show()

w_D=widgets.Text(value='1010',description='D3..D0:',layout=widgets.Layout(width='300px'))
w_sel=widgets.IntSlider(value=0,min=0,max=3,description='sel:')
display(widgets.VBox([w_D,w_sel]),widgets.interactive_output(draw_mux,{'D_str':w_D,'sel':w_sel}))


Output()

## Demultiplexer — One Input Fans Out to a Chosen Output

The DEMUX is the mirror image: a single input is routed to exactly one of $N$ outputs selected by the address; all other outputs stay at 0. The lit path shows where the input is being delivered.

$$Y_i = \begin{cases} D & i = \text{sel} \\ 0 & \text{otherwise}\end{cases}$$


In [3]:
def draw_demux(D, sel):
    N=4
    Y=[D if i==sel else 0 for i in range(N)]
    fig,ax=plt.subplots(figsize=(7,4)); ax.set_xlim(0,8); ax.set_ylim(0,5); ax.axis('off')
    body=Polygon([(3,1.5),(5,0.5),(5,4.5),(3,3.5)],closed=True,fc='#eef2f7',ec='#34495e',lw=1.6,zorder=2)
    ax.add_patch(body)
    ax.text(4,2.5,f'1:{N}\nDEMUX',ha='center',va='center',fontsize=9,weight='bold',zorder=3)
    ax.scatter([1.0],[2.5],s=44,color=wcol(D),zorder=4)
    ax.text(0.7,2.5,f'D={D}',ha='right',va='center',color=wcol(D),fontsize=10,weight='bold')
    ax.plot([1.0,3.0],[2.5,2.5],color=wcol(D),lw=wlw(D),zorder=1)
    ys=np.linspace(4.0,1.0,N)
    for i in range(N):
        active=(i==sel)
        col=wcol(Y[i]) if active else '#dddddd'; lw=wlw(Y[i]) if active else 1.0
        if active: ax.plot([4.0,5.0],[2.5,ys[i]],color=wcol(Y[i]),lw=wlw(max(Y[i],1)),zorder=3)
        ax.plot([5.0,6.6],[ys[i],ys[i]],color=col,lw=lw,zorder=1)
        ax.scatter([6.6],[ys[i]],s=40,color=wcol(Y[i]),zorder=4)
        ax.text(6.85,ys[i],f'Y{i}={Y[i]}',ha='left',va='center',color=wcol(Y[i]),fontsize=9,weight='bold')
    nbits=int(np.log2(N))
    ax.text(4,0.2,'sel = '+format(sel,f'0{nbits}b')+f' (={sel})',ha='center',fontsize=9,color='#8e44ad',weight='bold')
    ax.set_title(f'1:4 DEMUX  --  input delivered to Y{sel}',fontsize=10)
    plt.tight_layout(); plt.show()
w_Dd=widgets.ToggleButtons(options=[0,1],value=1,description='D:')
w_seld=widgets.IntSlider(value=2,min=0,max=3,description='sel:')
display(widgets.VBox([w_Dd,w_seld]),widgets.interactive_output(draw_demux,{'D':w_Dd,'sel':w_seld}))


Output()

## Decoder — Activate One of 2^n Output Lines

An $n$-to-$2^n$ decoder asserts exactly the output line whose index equals the binary input. It is a DEMUX with the data input tied high, and the basis of address decoding in memory. The asserted line lights up.

$$Y_i = 1 \iff \text{input} = i$$


In [4]:
def draw_decoder(nbits, value):
    N=2**nbits; value%=N
    Y=[1 if i==value else 0 for i in range(N)]
    fig,ax=plt.subplots(figsize=(7,0.6*N+1.5)); ax.set_xlim(0,8); ax.set_ylim(0,N+1); ax.axis('off')
    ax.add_patch(Rectangle((2.5,0.8),1.8,N,fc='#eef2f7',ec='#34495e',lw=1.6,zorder=2))
    ax.text(3.4,0.8+N/2,f'{nbits}:{N}\ndecoder',ha='center',va='center',fontsize=9,weight='bold',zorder=3)
    # input bits
    sbits=[(value>>b)&1 for b in range(nbits)]
    for b in range(nbits):
        yy=0.8+N - (b+1)*N/(nbits+1)
        ax.scatter([1.2],[yy],s=38,color=wcol(sbits[nbits-1-b]),zorder=4)
        ax.text(0.9,yy,f'A{nbits-1-b}={sbits[nbits-1-b]}',ha='right',va='center',color=wcol(sbits[nbits-1-b]),fontsize=9,weight='bold')
        ax.plot([1.2,2.5],[yy,yy],color=wcol(sbits[nbits-1-b]),lw=wlw(sbits[nbits-1-b]),zorder=1)
    for i in range(N):
        yy=0.8+N-(i+0.5)*N/N
        ax.plot([4.3,6.0],[yy,yy],color=wcol(Y[i]),lw=wlw(Y[i]),zorder=1)
        ax.scatter([6.0],[yy],s=36,color=wcol(Y[i]),zorder=4)
        ax.text(6.2,yy,f'Y{i}={Y[i]}',ha='left',va='center',color=wcol(Y[i]),fontsize=9,weight='bold')
    ax.set_title(f'{nbits}-to-{N} decoder  --  input {value} asserts Y{value}',fontsize=10)
    plt.tight_layout(); plt.show()
w_nb=widgets.IntSlider(value=2,min=2,max=3,description='addr bits:')
w_val=widgets.IntSlider(value=0,min=0,max=7,description='input:')
display(widgets.VBox([w_nb,w_val]),widgets.interactive_output(draw_decoder,{'nbits':w_nb,'value':w_val}))


Output()

## BCD-to-7-Segment Decoder — Driving a Real Display

A specialised decoder maps a 4-bit BCD digit to the seven segment-enable lines $a..g$ of a numeric display. Below, the digit you select lights the matching segments; this is the everyday face of decoder logic.


In [ ]:
SEG = {
  0:'1111110',1:'0110000',2:'1101101',3:'1111001',4:'0110011',
  5:'1011011',6:'1011111',7:'1110000',8:'1111111',9:'1111011'}
def draw_7seg(digit):
    s=SEG[digit]  # a,b,c,d,e,f,g
    on=[c=='1' for c in s]
    fig,ax=plt.subplots(figsize=(3.4,4.6)); ax.set_xlim(0,2); ax.set_ylim(0,3.4); ax.axis('off')
    def seg(coords,lit): ax.plot(*zip(*coords),color='#c0392b' if lit else '#eee',lw=7,solid_capstyle='round')
    seg([(0.4,3.0),(1.6,3.0)],on[0])   # a
    seg([(1.6,3.0),(1.6,1.8)],on[1])   # b
    seg([(1.6,1.8),(1.6,0.6)],on[2])   # c
    seg([(0.4,0.6),(1.6,0.6)],on[3])   # d
    seg([(0.4,1.8),(0.4,0.6)],on[4])   # e
    seg([(0.4,3.0),(0.4,1.8)],on[5])   # f
    seg([(0.4,1.8),(1.6,1.8)],on[6])   # g
    labels=' '.join(f'{n}={int(v)}' for n,v in zip('abcdefg',on))
    ax.set_title(f'digit {digit}\n{labels}',fontsize=9)
    plt.tight_layout(); plt.show()
w_dig=widgets.IntSlider(value=8,min=0,max=9,description='BCD digit:')
display(w_dig,widgets.interactive_output(draw_7seg,{'digit':w_dig}))


IntSlider(value=8, description='BCD digit:', max=9)

Output()

## Encoder and Priority Encoder — Many Lines to a Binary Code

An encoder outputs the binary index of an asserted input. A plain encoder misbehaves if two inputs are high; a **priority encoder** resolves this by reporting the highest-priority active line and asserting a *valid* flag. The active input and the resulting code light up.

$$\text{out} = \max\{\, i : D_i = 1 \,\}, \qquad V = \bigvee_i D_i$$


In [6]:
def draw_priority(D_str):
    N=8
    D=[int(c) for c in D_str.ljust(N,'0')[:N]]
    active=[i for i in range(N) if D[i]]
    valid=1 if active else 0
    code=max(active) if active else 0
    fig,ax=plt.subplots(figsize=(7.5,4)); ax.set_xlim(0,9); ax.set_ylim(0,N+1); ax.axis('off')
    ax.add_patch(Rectangle((3,0.6),2,N,fc='#eef2f7',ec='#34495e',lw=1.6,zorder=2))
    ax.text(4,0.6+N/2,'8:3\npriority\nencoder',ha='center',va='center',fontsize=8.5,weight='bold',zorder=3)
    for i in range(N):
        yy=N-i
        chosen=(i==code and valid)
        ax.scatter([1.4],[yy],s=38,color=wcol(D[i]),zorder=4)
        tag=' <-- highest' if chosen else ''
        ax.text(1.1,yy,f'D{i}={D[i]}',ha='right',va='center',color=wcol(D[i]),fontsize=9,weight='bold')
        col=wcol(D[i]) if not chosen else '#8e44ad'; lw=2.6 if chosen else wlw(D[i])
        ax.plot([1.4,3.0],[yy,yy],color=col,lw=lw,zorder=1)
        if chosen: ax.text(2.0,yy+0.25,'win',fontsize=7,color='#8e44ad')
    cbits=[(code>>b)&1 for b in range(3)]
    for b in range(3):
        yy=N-1.5-b*1.5
        ax.plot([5,6.4],[yy,yy],color=wcol(cbits[2-b]),lw=wlw(cbits[2-b]),zorder=1)
        ax.scatter([6.4],[yy],s=36,color=wcol(cbits[2-b]),zorder=4)
        ax.text(6.6,yy,f'Q{2-b}={cbits[2-b]}',ha='left',va='center',color=wcol(cbits[2-b]),fontsize=9,weight='bold')
    ax.plot([5,6.4],[1.0,1.0],color=wcol(valid),lw=wlw(valid),zorder=1)
    ax.scatter([6.4],[1.0],s=36,color=wcol(valid),zorder=4)
    ax.text(6.6,1.0,f'V={valid}',ha='left',va='center',color=wcol(valid),fontsize=9,weight='bold')
    ax.set_title(f'priority encoder  --  code={code} ({format(code,"03b")}), valid={valid}',fontsize=10)
    plt.tight_layout(); plt.show()
w_pe=widgets.Text(value='00100110',description='D0..D7:',layout=widgets.Layout(width='320px'))
display(w_pe,widgets.interactive_output(draw_priority,{'D_str':w_pe}))


Text(value='00100110', description='D0..D7:', layout=Layout(width='320px'))

Output()

## A MUX Implements Any Truth Table

Tie the data inputs of a $2^n{:}1$ MUX to the rows of a truth table and the select lines to the variables: the MUX outputs that function directly. This makes the multiplexer a universal lookup element. The cell verifies an arbitrary 2-variable function against its MUX realisation.


In [7]:
def mux_as_function(truth_str):
    # truth_str: outputs for (A,B) = 00,01,10,11
    rows=[int(c) for c in truth_str.ljust(4,'0')[:4]]
    print('implementing function with a 4:1 MUX (selects = A,B):')
    print(f"{'A':>2}{'B':>3} | MUX out | direct")
    ok=True
    for A in (0,1):
        for B in (0,1):
            sel=A*2+B
            mux_out=rows[sel]
            print(f'{A:>2}{B:>3} | {mux_out:>7} | {rows[sel]:>6}')
    print('\nMUX data inputs D0..D3 =',rows,'-> reproduces the table exactly')

w_tt=widgets.Text(value='0110',description='truth (00,01,10,11):',layout=widgets.Layout(width='420px'),style={'description_width':'160px'})
display(w_tt,widgets.interactive_output(mux_as_function,{'truth_str':w_tt}))


Text(value='0110', description='truth (00,01,10,11):', layout=Layout(width='420px'), style=TextStyle(descripti…

Output()